# Training Notebook

This notebook demonstrates the full pipeline for converting your videos into an HDF5 dataset. The steps include:

1. Setting parameters (resolution and fps).
2. Converting MOV files to MP4 using the provided shell script (`convert_mov_to_mp4.sh`).
3. Converting videos into image frames (dataset acquisition).
4. Converting the image frames into an HDF5 dataset.

Each step includes outputs (e.g., directory listings) so that you can see what is happening at each stage.

In [ ]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

## Step 1: Set Parameters

We are using a target resolution of **640x480** and a frame rate of **30fps**.

In [ ]:
resolution = "640x480"
fps = 30
print("Using resolution:", resolution, "and fps:", fps)

## Step 2: Convert MOV Files to MP4

The shell script `convert_mov_to_mp4.sh` (located in `example/training/`) converts all `.MOV` files in the `videos` folder into `.mp4` files using ffmpeg. Here is the content of the script:

```bash
#!/bin/bash
cd videos
for f in *.MOV; do
  output="${f%.*}.mp4"
  ffmpeg -i "$f" -c:v libx264 -c:a aac "$output"
  # rm "$f"
done
```

Now, let's run the script.

In [ ]:
!./convert_mov_to_mp4.sh

# List the files in the videos directory to see the new MP4 files
!ls videos

## Step 3: Convert Videos to Image Dataset

Next, we convert the videos into an image dataset by extracting frames. The command below runs the module `dataset.acquisition.convert_video_directory` with the specified parameters.

The images will be stored in `example/training/data_dir/train`.

In [ ]:
from dataset.acquisition.convert_video_directory import run_video_conversion
run_video_conversion("videos", "data_dir/train", (640,480), fps=30, override=True, video_extension="MOV", quiet=True)

# Verify by listing the output directory
!ls data_dir/train

## Step 4: Convert Image Dataset to HDF5

Finally, we convert the extracted images to an HDF5 dataset using `dataset/convert_to_h5.py`. The output will be stored in the directory `example/training/hdf5_dataset`.

In [ ]:
from dataset.convert_to_h5 import run_make_h5
run_make_h5("data_dir", "hdf5_dataset", target_size=(640,480), extension="png", override=True)


# List the contents of the hdf5_dataset directory to verify
!ls hdf5_dataset

## Summary

You have now converted your MOV videos into MP4 files, extracted frames to create a training dataset, and built an HDF5 dataset from those frames. This notebook provides a clear, step-by-step process with immediate feedback at each stage.

Feel free to modify or expand this notebook as needed for further experimentation or integration into your workflow.